# 向量、矩阵与运算（Vectors, Matrices & Operations）

对应课程：`phases/01-math-foundations/02-vectors-matrices-operations`

> 每个神经网络都只是多绕了几步的矩阵乘法。

本 notebook 把 `matrices.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `matrices.py`。

**贯穿全课的模式：** 一层就是 `relu(W @ x + b)`。形状对得上才能乘；广播把偏置铺开；行列式决定能不能走回去。


## 0. 依赖

只用标准库。`matrices.py` 里的 `random` 只给随机初始化用，本 notebook 不需要。


In [1]:
# 无第三方依赖


## 1. Vector（短版）

矩阵课里的向量只保留加减、标量乘、点积、模长、归一化。`[3, 4]` 模长仍是 5。


In [2]:
class Vector:
    def __init__(self, data):
        """有序数字列表。"""
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self, other):
        """逐元素相加。"""
        return Vector([a + b for a, b in zip(self.data, other.data)])

    def __sub__(self, other):
        """逐元素相减。"""
        return Vector([a - b for a, b in zip(self.data, other.data)])

    def __mul__(self, scalar):
        """标量乘法。"""
        return Vector([x * scalar for x in self.data])

    def dot(self, other):
        """点积。"""
        return sum(a * b for a, b in zip(self.data, other.data))

    def magnitude(self):
        """欧几里得长度。"""
        return sum(x ** 2 for x in self.data) ** 0.5

    def normalize(self):
        """单位向量。"""
        mag = self.magnitude()
        return Vector([x / mag for x in self.data])


v = Vector([3, 4])
w = Vector([1, 2])
print("v =", v)
print("v + w =", v + w)
print("v · w =", v.dot(w))
print("|v| =", v.magnitude())


v = Vector([3, 4])
v + w = Vector([4, 6])
v · w = 11
|v| = 5.0


## 2. Matrix：形状、加法、广播

矩阵是 $m\times n$ 的数字网格。同形状逐元素相加；一行偏置可以对每一行广播——这就是 `output + bias`。

notebook 里 `__repr__` 只打印列表，避免课程禁止的 ASCII 框线。


In [3]:
class Matrix:
    def __init__(self, data):
        """行列表。shape = (行数, 列数)。"""
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self):
        """朴素表示，不用 Unicode 框线。"""
        return f"Matrix({self.data})"

    def __add__(self, other):
        """同形状逐元素加；1 行或 1 列则广播。"""
        if isinstance(other, Matrix):
            if other.shape == self.shape:
                return Matrix([
                    [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
                    for i in range(self.rows)
                ])
            if other.rows == 1 and other.cols == self.cols:
                return Matrix([
                    [self.data[i][j] + other.data[0][j] for j in range(self.cols)]
                    for i in range(self.rows)
                ])
            if other.cols == 1 and other.rows == self.rows:
                return Matrix([
                    [self.data[i][j] + other.data[i][0] for j in range(self.cols)]
                    for i in range(self.rows)
                ])
        raise ValueError(f"Cannot add shapes {self.shape} and {other.shape}")


A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
print("A + B =", A + B)
out = Matrix([[1, 2, 3], [4, 5, 6]])
bias = Matrix([[10, 20, 30]])
print("2x3 + 1x3 广播 =", out + bias)


A + B = Matrix([[6, 8], [10, 12]])
2x3 + 1x3 广播 = Matrix([[11, 22, 33], [14, 25, 36]])


## 3. 矩阵乘法

$$
(m\times n)\,@\,(n\times p) = (m\times p),\qquad
C_{ij} = \sum_k A_{ik} B_{kj}
$$

内层维度必须相等，否则形状对不上。`2x3 @ 3x2` 得到 `2x2`。


In [4]:
def matmul(self, other):
    """行点列。inner dim 必须匹配。"""
    if self.cols != other.rows:
        raise ValueError(
            f"Cannot multiply shapes {self.shape} and {other.shape}: "
            f"inner dimensions {self.cols} != {other.rows}"
        )
    return Matrix([
        [
            sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
            for j in range(other.cols)
        ]
        for i in range(self.rows)
    ])


def __matmul__(self, other):
    return self.matmul(other)


Matrix.matmul = matmul
Matrix.__matmul__ = __matmul__

A = Matrix([[1, 2, 3], [4, 5, 6]])
B = Matrix([[7, 8], [9, 10], [11, 12]])
C = A @ B
print("A shape", A.shape, "@ B shape", B.shape, "->", C.shape)
print("A @ B =", C.data)
print("期望 [[58, 64], [139, 154]]")


A shape (2, 3) @ B shape (3, 2) -> (2, 2)
A @ B = [[58, 64], [139, 154]]
期望 [[58, 64], [139, 154]]


## 4. 转置

$$
(A^\top)_{ij} = A_{ji}
$$

$m\times n$ 变成 $n\times m$。反向传播里梯度要从输出侧走回输入侧，转置是换轴的标准动作。


In [5]:
def transpose(self):
    """行列互换。"""
    return Matrix([
        [self.data[j][i] for j in range(self.rows)]
        for i in range(self.cols)
    ])


Matrix.transpose = transpose
Matrix.T = property(lambda self: self.transpose())

A = Matrix([[1, 2, 3], [4, 5, 6]])
print("A =", A.data, "shape", A.shape)
print("A.T =", A.T.data, "shape", A.T.shape)


A = [[1, 2, 3], [4, 5, 6]] shape (2, 3)
A.T = [[1, 4], [2, 5], [3, 6]] shape (3, 2)


## 5. 行列式

$2\times 2$：

$$
\det\begin{bmatrix}a&b\\c&d\end{bmatrix} = ad-bc
$$

更大的方阵按第一行展开成余子式。$\det=0$ 表示列线性相关，空间被压扁，不可逆。`[[1,2],[3,4]]` 的行列式是 $-2$。


In [6]:
def determinant(self):
    """只对方阵。2x2 用 ad-bc，更大的递归展开。"""
    if self.rows != self.cols:
        raise ValueError("Determinant only defined for square matrices")
    if self.shape == (1, 1):
        return self.data[0][0]
    if self.shape == (2, 2):
        return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
    det = 0
    for j in range(self.cols):
        minor = Matrix([
            [self.data[i][k] for k in range(self.cols) if k != j]
            for i in range(1, self.rows)
        ])
        det += ((-1) ** j) * self.data[0][j] * minor.determinant()
    return det


Matrix.determinant = determinant

M = Matrix([[1, 2], [3, 4]])
print("det([[1,2],[3,4]]) =", M.determinant())


det([[1,2],[3,4]]) = -2


## 6. $2\times 2$ 逆

$$
\begin{bmatrix}a&b\\c&d\end{bmatrix}^{-1}
= \frac{1}{ad-bc}
\begin{bmatrix}d&-b\\-c&a\end{bmatrix}
$$

$|\det|$ 太小视为奇异。`A @ A^{-1}` 应回到单位阵。


In [7]:
def inverse_2x2(self):
    """2x2 解析逆。奇异矩阵抛错。"""
    if self.shape != (2, 2):
        raise ValueError("This method only works for 2x2 matrices")
    det = self.determinant()
    if abs(det) < 1e-10:
        raise ValueError("Matrix is singular, no inverse exists")
    return Matrix([
        [self.data[1][1] / det, -self.data[0][1] / det],
        [-self.data[1][0] / det, self.data[0][0] / det],
    ])


Matrix.inverse_2x2 = inverse_2x2

A = Matrix([[4, 7], [2, 6]])
A_inv = A.inverse_2x2()
print("A =", A.data)
print("det(A) =", A.determinant())
print("A^{-1} =", A_inv.data)
prod = A @ A_inv
print("A @ A^{-1} =", [[round(x, 10) for x in row] for row in prod.data])


A = [[4, 7], [2, 6]]
det(A) = 10
A^{-1} = [[0.6, -0.7], [-0.2, 0.4]]
A @ A^{-1} = [[1.0, 0.0], [-0.0, 1.0]]


## 7. 单位阵与零矩阵

$$
I_{ij} = [i=j],\qquad (I A = A I = A)
$$

单位阵什么都不做，残差连接里的「加上自己」就是它。`zeros` 用来占位偏置。


In [8]:
@staticmethod
def identity(n):
    """n x n 单位阵。"""
    return Matrix([
        [1 if i == j else 0 for j in range(n)]
        for i in range(n)
    ])


@staticmethod
def zeros(rows, cols):
    """全零矩阵。"""
    return Matrix([[0] * cols for _ in range(rows)])


Matrix.identity = identity
Matrix.zeros = zeros

A = Matrix([[4, 7], [2, 6]])
I = Matrix.identity(2)
print("I =", I.data)
print("I @ A =", (I @ A).data)
print("A @ I =", (A @ I).data)
print("zeros(2,3) =", Matrix.zeros(2, 3).data)


I = [[1, 0], [0, 1]]
I @ A = [[4, 7], [2, 6]]
A @ I = [[4, 7], [2, 6]]
zeros(2,3) = [[0, 0, 0], [0, 0, 0]]


## 8. ReLU

$$
\mathrm{ReLU}(x) = \max(0, x)
$$

逐元素把负数打成 0。一层前向就是 `relu(W @ x + b)` 里的非线性。


In [9]:
def relu_matrix(m):
    """逐元素 max(0, ·)。"""
    return Matrix([[max(0, val) for val in row] for row in m.data])


Z = Matrix([[1.0, -2.0, 0.5], [-0.3, 4.0, -5.0]])
print("z =", Z.data)
print("relu(z) =", relu_matrix(Z).data)


z = [[1.0, -2.0, 0.5], [-0.3, 4.0, -5.0]]
relu(z) = [[1.0, 0, 0.5], [0, 4.0, 0]]


## 对照表

| 函数 | 角色 |
|------|------|
| `Vector.*` | 短向量：加减、点积、模长 |
| `Matrix.__add__` | 逐元素加；一行/一列广播偏置 |
| `Matrix.matmul` / `@` | $(m\times n)(n\times p)=m\times p$ |
| `Matrix.transpose` / `.T` | 行列互换 |
| `Matrix.determinant` | 体积缩放；0 则不可逆 |
| `Matrix.inverse_2x2` | 撤销 $2\times 2$ 变换 |
| `Matrix.identity` / `zeros` | 什么都不做 / 占位 |
| `relu_matrix` | 逐元素 $\max(0,x)$ |

要看完整打印 demo，运行：

```bash
python matrices.py
```
